In [8]:
import sys
sys.path.append("..")


In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from factor_engines.load_data import get_price_data, compute_returns
from factor_engines.factors import (
    momentum_factor,
    volatility_factor,
    reversal_factor,
    sma_distance_factor,
    compute_all,
    zscore,
    daily_score
)
from factor_engines.portfolio import (
    build_long_short_portfolio,
    build_zscore_portfolio_from_factor,
    build_composite_portfolio,
    composite_factor
)
from factor_engines.regression import (
    summarize_regression
)

plt.style.use("seaborn-v0_8")


In [10]:
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "META"]

prices = get_price_data(tickers, "2015-01-01", "2025-01-01")
returns = compute_returns(prices["Close"])




In [11]:
momentum = momentum_factor(prices)
volatility = volatility_factor(prices)
reversal = reversal_factor(prices)
sma = sma_distance_factor(prices)

factors = {
    "momentum": momentum,
    "volatility": volatility,
    "reversal": reversal,
    "sma_distance": sma
}


In [12]:
mom_weights_ls, mom_ret_ls = build_long_short_portfolio(momentum, returns)
mom_weights_z, mom_ret_z = build_zscore_portfolio_from_factor(momentum, returns)
comp_weights, comp_ret = build_composite_portfolio(factors, returns)


In [13]:
market_prices = get_price_data(["SPY"], "2015-01-01", "2025-01-01")
close_prices = market_prices["Close"]
spy_prices = close_prices["SPY"]
market_ret = compute_returns(spy_prices)

market_ret.head()


Date
2015-01-05   -0.018060
2015-01-06   -0.009419
2015-01-07    0.012461
2015-01-08    0.017745
2015-01-09   -0.008013
Name: SPY, dtype: float64

In [14]:
comp_weights.sum(axis=1).head(30)


Date
2016-01-04   -6.938894e-17
2016-01-05   -5.605437e-02
2016-01-06   -1.747060e-02
2016-01-07   -3.267819e-02
2016-01-08   -6.245005e-17
2016-01-11    5.551115e-17
2016-01-12   -5.080106e-02
2016-01-13    3.469447e-17
2016-01-14    5.551115e-17
2016-01-15   -1.387779e-17
2016-01-19    2.775558e-17
2016-01-20    1.387779e-17
2016-01-21   -5.551115e-17
2016-01-22    8.326673e-17
2016-01-25   -1.214011e-02
2016-01-26   -1.387779e-17
2016-01-27    0.000000e+00
2016-01-28   -4.861913e-02
2016-01-29   -9.442363e-03
2016-02-01    2.775558e-17
2016-02-02    1.523345e-02
2016-02-03    4.878910e-17
2016-02-04   -4.770490e-17
2016-02-05   -1.784011e-02
2016-02-08   -5.343817e-03
2016-02-09   -2.550366e-03
2016-02-10   -1.867634e-02
2016-02-11   -5.637851e-17
2016-02-12   -2.974303e-03
2016-02-16   -6.005799e-02
dtype: float64

In [15]:
combined = pd.concat(
    [comp_ret, market_ret],
    axis=1,
    keys=["comp", "SPY"],
    join="inner"
)

combined.head(20), combined.shape


(                comp       SPY
 Date                          
 2016-01-04 -0.015237 -0.013979
 2016-01-05  0.003548  0.001692
 2016-01-06  0.006088 -0.012614
 2016-01-07  0.003087 -0.023991
 2016-01-08 -0.001565 -0.010977
 2016-01-11  0.001190  0.000990
 2016-01-12 -0.005072  0.008068
 2016-01-13 -0.010864 -0.024941
 2016-01-14 -0.001202  0.016417
 2016-01-15 -0.002934 -0.021467
 2016-01-19  0.005699  0.001332
 2016-01-20 -0.002110 -0.012815
 2016-01-21  0.001539  0.005602
 2016-01-22 -0.000812  0.020515
 2016-01-25  0.002184 -0.015116
 2016-01-26  0.000106  0.013643
 2016-01-27  0.007691 -0.010883
 2016-01-28 -0.002042  0.005209
 2016-01-29 -0.048137  0.024377
 2016-02-01 -0.000227 -0.000361,
 (2264, 2))

In [16]:
from factor_engines.regression import summarize_regression

summary_comp, results_comp = summarize_regression(comp_ret, market_ret)
summary_comp


{'alpha_daily': np.float64(0.00010380579446673853),
 'alpha_annual': np.float64(0.02615906020561811),
 'alpha_tstat': np.float64(1.0981736891449976),
 'r2': np.float64(0.0037336223145878478),
 'beta_SPY': 0.02445544761121559,
 'beta_SPY_tstats': 2.911546664606476}